# Observability: Metrics and Logs with Grafana and LokiStack

This notebook explores the pre-deployed observability stack for the Custom Deep Research Lab.
The lab uses **Grafana** for dashboarding Prometheus metrics and **LokiStack** (via the
Cluster Observability Operator) for centralized log aggregation. You will verify the
observability components, review the key agentic metrics, and learn how to write LogQL
queries to inspect application behavior from the OpenShift Console.

## 1. Prerequisites

In [ ]:
import subprocess

ns = "doc-research-lab"
result = subprocess.run(["oc", "get", "namespace", ns], capture_output=True, text=True)
if result.returncode == 0:
    print(f"\u2705 Namespace '{ns}' exists")
else:
    print(f"\u274c Namespace '{ns}' not found \u2014 deploy the application first")

## 2. Verify Observability Stack

In [ ]:
import subprocess

checks = [
    ("LokiStack", "oc get lokistack -A -o name"),
    ("ServiceMonitor", f"oc get servicemonitor research-backend -n {ns} -o name"),
    ("GrafanaDashboard", f"oc get grafanadashboard research-lab-metrics -n {ns} -o name"),
]
for name, cmd in checks:
    result = subprocess.run(cmd.split(), capture_output=True, text=True)
    status = "\u2705" if result.returncode == 0 else "\u26a0\ufe0f Not found"
    print(f"  {status} {name}")

## 3. Key Agentic Metrics (Prometheus)

The research backend exposes Prometheus metrics on `/metrics`. These are scraped by the
`ServiceMonitor` and visualized in the Grafana dashboard.

| # | Metric | Prometheus Name | Description |
|---|--------|----------------|-------------|
| 1 | Request Rate | `research_requests_total` | Total number of research requests received |
| 2 | Latency Percentiles | `research_request_duration_seconds` | End-to-end request latency (p50, p90, p99) |
| 3 | LLM Token Usage | `llm_tokens_total` | Token consumption by model and type (input/output) |
| 4 | LLM Inference Latency | `llm_inference_duration_seconds` | Time spent in LLM inference calls |
| 5 | Tool Call Success Rate | `tool_calls_total` | MCP tool invocations by name and status (success/error) |
| 6 | Research Quality Scores | `research_quality_score` | LLM-as-Judge quality scores per iteration |

These metrics are sufficient to monitor the health and efficiency of any agentic application.

## 4. Explore the Grafana Dashboard

The `research-lab-metrics` GrafanaDashboard CR deploys a pre-built dashboard.

**How to access:**
1. Open the OpenShift Console
2. Navigate to **Networking \u2192 Routes** in the `doc-research-lab` namespace
3. Click the Grafana route URL
4. Log in with your OpenShift credentials (OAuth proxy)
5. Select the **Research Lab Metrics** dashboard

**Dashboard panels:**

| Row | Panel | Metrics Used |
|-----|-------|--------------|
| Overview | Request Rate (req/min) | `rate(research_requests_total[5m])` |
| Overview | Latency Percentiles | `histogram_quantile(0.95, research_request_duration_seconds_bucket)` |
| Overview | Active Requests | `research_active_requests` |
| AI Agent | Token Usage by Model | `sum by(model)(rate(llm_tokens_total[5m]))` |
| AI Agent | LLM Inference Latency | `histogram_quantile(0.95, llm_inference_duration_seconds_bucket)` |
| AI Agent | Tool Call Success Rate | `sum(rate(tool_calls_total{status="success"}[5m]))` |
| AI Agent | Quality Scores Over Time | `research_quality_score` |

## 5. LogQL Queries

LokiStack automatically collects STDOUT and STDERR from all containers running in the
cluster. The research backend uses structured JSON logging, so you can parse and filter
fields directly in LogQL without any additional log shipping configuration.

**Where to run these queries:**
- OpenShift Console \u2192 **Observe \u2192 Logs**
- Or the Grafana Explore tab with the Loki data source

In [ ]:
ns = "doc-research-lab"

queries = [
    ("All application logs",
     f'{{ log_type="application", kubernetes_namespace_name="{ns}" }}'),
    ("Backend logs only",
     f'{{ log_type="application", kubernetes_namespace_name="{ns}", kubernetes_container_name="backend" }} | json | line_format "{{{{.message}}}}"'),
    ("Error logs",
     f'{{ log_type="application", kubernetes_namespace_name="{ns}" }} | json | level="ERROR"'),
    ("Verification results",
     f'{{ log_type="application", kubernetes_namespace_name="{ns}" }} | json | layer="verification"'),
    ("Search operations",
     f'{{ log_type="application", kubernetes_namespace_name="{ns}" }} | json | operation=~"search.*"'),
    ("Report generation rate",
     f'rate({{ log_type="application", kubernetes_namespace_name="{ns}" }} | json | operation="draft_report" [5m])'),
]

print("LogQL Queries for OpenShift Console \u2192 Observe \u2192 Logs:\n")
for desc, q in queries:
    print(f"\U0001f4cb {desc}:")
    print(f"   {q}\n")
print("\u2705 Copy these queries into the OpenShift Console log viewer")

## 6. Summary

In [ ]:
print("\u2705 Observability walkthrough complete")
print()
print("What we covered:")
print("  1. Verified the observability stack (LokiStack, ServiceMonitor, GrafanaDashboard)")
print("  2. Reviewed 6 key agentic Prometheus metrics")
print("  3. Explored the Grafana dashboard panels")
print("  4. Built LogQL queries for application log analysis")
print()
print("Next: 4_tracing_mlflow.ipynb \u2014 MLflow tracing for agent spans")